In [7]:
import joblib
import pandas as pd
import numpy as np

# defining the rules first

def rule_low_active(df): #Near-zero active time with high score
    return (df['active_days'] < 50) & (df['Score'] >= 10)

def rule_new_account_top_rank(df): #new user high rank
    return (df["ContestCnt"] <= 5) & (df["Rank"] < 200)

def rule_perfect_solve(df):
    return (df['total_fail'] == 0) & (df['Score'] >= 15) & (df['ContestCnt'] <= 5)

def rule_weak_profile(df): #Weak profile but top contest rank
    return (df['Total_solved'] < 200) & (df['Rank'] < 200)

def rule_low_solved_high_rating(df): #Low solved count but high rating
    return (df['Total_solved'] < 100) & (df['CurrRating'] > 1900)

def rule_zero_rating_jump(df):
    return (df['rating_jump'] == 0) & (df['CurrRating'] > 1500)

def rule_ghost_profile(df):
    return (df['Total_solved'] < 50) & (df['Rank'] < 5000)


# Loading Model
model = joblib.load("cheat_detection_model.pkl")

lof_models = model["lof_models"]
LOF_FEATURES = model["LOF_FEATURES"]
RATING_BINS = model["RATING_BINS"]
RATING_LABELS = model["RATING_LABELS"]
RULES = model["RULES"]
#efficiency_threshold = model["efficiency_threshold"]
threshold = model["Combined_threshold"]
lof_max = model["lof_max"]
lof_min = model["lof_min"]
print("Model loaded successfully")


RULE_FUNC_MAP = {
    'Near-zero active time with high score' : rule_low_active,
    'new user high rank'                    : rule_new_account_top_rank,
    'Perfect solve'                         : rule_perfect_solve,
    'Weak profile'                          : rule_weak_profile,
    'Low solved high rating'                : rule_low_solved_high_rating,
    # 'Extreme efficiency'                    : rule_extreme_efficiency,
    'Zero rating jump high rating'          : rule_zero_rating_jump,
    'Ghost profile high rank'               : rule_ghost_profile,
}

for rule in RULES:
    rule['func'] = RULE_FUNC_MAP[rule['name']]

# ── Predict Function ───────────────────────────────────────────────
def predict_user(user):
    df = pd.DataFrame([user])

    # Derived features
    df['score_per_time']     = df['CurrConSolved'] / df['time_taken'].clip(lower=1)
    df['hard_ratio']         = df['Hard'] / (df['Total_solved'] + 1)
    df['solved_per_contest'] = df['Total_solved'] / (df['ContestCnt'] + 1)
    df['rating_jump']        = df['CurrRating'] - df['rating_first']

    # Rating bucket
    df['rating_bucket'] = pd.cut(
        df['CurrRating'],
        bins=RATING_BINS,
        labels=RATING_LABELS
    )
    bucket = str(df['rating_bucket'].iloc[0])

    if bucket not in lof_models:
        return "Not enough data for this rating bucket"

    # LOF score
    scaler = lof_models[bucket]['scaler']
    lof    = lof_models[bucket]['lof']

    X        = df[LOF_FEATURES].fillna(0)
    X_scaled = scaler.transform(X)

    lof_score        = -lof.score_samples(X_scaled)[0]
    lof_score_clipped = min(lof_score, lof_max)          # clip to p99 to avoid compression
    lof_normalized   = ((lof_score_clipped - lof_min) / (lof_max - lof_min + 1e-9)) * 3

    # Rule score
    rule_score = 0
    triggered  = []
    for rule in RULES:
        if 'threshold' in rule:
            hit = rule['func'](df, rule['threshold']).iloc[0]
        else:
            hit = rule['func'](df).iloc[0]
        if hit:
            rule_score += rule['weight']
            triggered.append(rule['name'])
    if rule_score == 0:
        lof_normalized = 0

    final_score = rule_score + (lof_normalized * 0.3)
    thresh=threshold
    # print(thresh)
    if final_score >= 6:
        label = "HIGHLY SUSPICIOUS"
    elif final_score >= thresh:
        label = "SUSPICIOUS"
    else:
        label = "CLEAN"

    return {
        "lof_score":       round(lof_score, 4),
        "lof_normalized":  round(lof_normalized, 4),
        "rule_score":      rule_score,
        "final_score":     round(final_score, 4),
        "suspicious":      final_score >= thresh,
        "label":           label,
        "triggered_rules": triggered
    }

#suspicious
user1 = {
    "Rank":1,
    "Username":"sunnynguyenai",
    "Score":18,
    "CurrConSolved":4,
    "time_taken":502,
    "total_fail":0,
    "Country":"United States",
    "Profile_rank":870517,
    "PostViewCnt":53,
    "Reputation":15,
    "Total_solved":178,
    "Easy":65,
    "Medium":90,
    "Hard":23,
    "ContestCnt":7,
    "CurrRating":2347,
    "rating_first":1414,
    "GlobalRank":3415,
    "active_days":161,
}
#clean
user2 = {
    "Rank": 21,
    "Username": "darksparrow9932",
    "Score": 18,
    "CurrConSolved": 4,
    "time_taken": 875,
    "total_fail": 0,
    "Country": "India",
    "Profile_rank": 29654,
    "PostViewCnt": 31,
    "Reputation": 0,
    "Total_solved": 1006,
    "Easy": 262,
    "Medium": 569,
    "Hard": 175,
    "ContestCnt": 13,
    "CurrRating": 2068,
    "rating_first": 1486,
    "GlobalRank": 14595,
    "active_days": 518
}
#clean
user3 = {
    "Rank": 16,
    "Username": "VeritasVelata",
    "Score": 18,
    "CurrConSolved": 4,
    "time_taken": 835,
    "total_fail": 0,
    "Country": "Sanctuary",
    "Profile_rank": 19,
    "PostViewCnt": 18701,
    "Reputation": 137,
    "Total_solved": 3893,
    "Easy": 935,
    "Medium": 2037,
    "Hard": 921,
    "ContestCnt": 23,
    "CurrRating": 3321,
    "rating_first": 1886,
    "GlobalRank": 25,
    "active_days": 148
}
#highly suspicious
user4 = {
    "Rank": 4,
    "Username": "Mona0405",
    "Score": 20,
    "CurrConSolved": 4,
    "time_taken": 552,
    "total_fail": 0,
    "Country": "",  # missing in data
    "Profile_rank": 1557154,
    "PostViewCnt": 0,
    "Reputation": 0,
    "Total_solved": 95,
    "Easy": 24,
    "Medium": 48,
    "Hard": 23,
    "ContestCnt": 1,
    "CurrRating": 2058,
    "rating_first": 2058,
    "GlobalRank": 15295,
    "active_days": 0
}



from datetime import datetime
from zoneinfo import ZoneInfo

#function to calculate start time
def convert_to_8am_same_day(timestamp):
    ist = ZoneInfo("Asia/Kolkata")
    
    # convert timestamp to IST
    dt = datetime.fromtimestamp(timestamp, ist)
    
    # set time to 8:00 AM same day
    dt_8am = dt.replace(hour=8, minute=0, second=0, microsecond=0)
    
    # return unix timestamp
    return int(dt_8am.timestamp())

import json

def extract_features(row):
    history = json.loads(row[30])
    history = sorted(history, key=lambda x: x["contest"]["startTime"])

    first_time = history[0]["contest"]["startTime"]
    last_time  = history[-1]["contest"]["startTime"]

    start_time = convert_to_8am_same_day(row[5])

    new_user = {
        "Rank": row[0],
        "Username": row[1],
        "Score": row[2],
        "CurrConSolved": row[9],
        "time_taken": row[5] - start_time,
        "total_fail": (row[9] or 0) + (row[10] or 0) + (row[11] or 0) + (row[12] or 0),
        "Country": row[14],
        "Profile_rank": row[15],
        "PostViewCnt": row[16],
        "Reputation": row[17],
        "Total_solved": row[18],
        "Easy": row[20],
        "Medium": row[22],
        "Hard": row[24],
        "ContestCnt": row[26],
        "CurrRating": row[27],
        "rating_first": history[0]["rating"],
        "GlobalRank": row[28],
        "active_days": (last_time - first_time) // 86400
    }

    return new_user

data = [1, 'SAIKRISHNAKANTHS', 18, 4, 1774149597, 1774147370, 1774147655, None, 1774148697, 3, 0, None, 0, 'SAI KRISHNA KANTH S', 'India', 217712, 0, 0, 462, 484, 204, 217, 198, 206, 60, 61, 27, 1544.366385991523, 282046, 33.0, '[{"attended": true, "trendDirection": "DOWN", "problemsSolved": 1, "ranking": 11475, "rating": 1486.496, "contest": {"title": "Weekly Contest 422", "startTime": 1730601000}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 1, "ranking": 13927, "rating": 1466.739, "contest": {"title": "Weekly Contest 423", "startTime": 1731205800}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 0, "ranking": 22110, "rating": 1413.716, "contest": {"title": "Weekly Contest 424", "startTime": 1731810600}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 0, "ranking": 19866, "rating": 1367.657, "contest": {"title": "Weekly Contest 425", "startTime": 1732415400}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 0, "ranking": 20498, "rating": 1326.396, "contest": {"title": "Weekly Contest 426", "startTime": 1733020200}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 0, "ranking": 27010, "rating": 1289.034, "contest": {"title": "Weekly Contest 440", "startTime": 1741487400}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 0, "ranking": 30278, "rating": 1254.016, "contest": {"title": "Weekly Contest 464", "startTime": 1756002600}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 16101, "rating": 1268.541, "contest": {"title": "Weekly Contest 466", "startTime": 1757212200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 15620, "rating": 1275.95, "contest": {"title": "Weekly Contest 470", "startTime": 1759631400}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 14926, "rating": 1297.287, "contest": {"title": "Weekly Contest 471", "startTime": 1760236200}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 1, "ranking": 19849, "rating": 1271.541, "contest": {"title": "Weekly Contest 473", "startTime": 1761445800}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 13486, "rating": 1277.477, "contest": {"title": "Weekly Contest 478", "startTime": 1764469800}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 10835, "rating": 1302.001, "contest": {"title": "Weekly Contest 479", "startTime": 1765074600}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 16346, "rating": 1307.288, "contest": {"title": "Weekly Contest 481", "startTime": 1766284200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 13661, "rating": 1322.567, "contest": {"title": "Weekly Contest 482", "startTime": 1766889000}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 18341, "rating": 1322.971, "contest": {"title": "Weekly Contest 484", "startTime": 1768098600}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 11528, "rating": 1354.281, "contest": {"title": "Weekly Contest 485", "startTime": 1768703400}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 18872, "rating": 1361.544, "contest": {"title": "Weekly Contest 487", "startTime": 1769913000}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 21016, "rating": 1365.405, "contest": {"title": "Weekly Contest 488", "startTime": 1770517800}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 2, "ranking": 11164, "rating": 1402.696, "contest": {"title": "Weekly Contest 489", "startTime": 1771122600}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 2, "ranking": 15308, "rating": 1419.063, "contest": {"title": "Weekly Contest 490", "startTime": 1771727400}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 1, "ranking": 25649, "rating": 1400.008, "contest": {"title": "Weekly Contest 491", "startTime": 1772332200}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 1, "ranking": 18847, "rating": 1405.946, "contest": {"title": "Weekly Contest 492", "startTime": 1772937000}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 2, "ranking": 7193, "rating": 1459.475, "contest": {"title": "Weekly Contest 493", "startTime": 1773541800}}, {"attended": true, "trendDirection": "UP", "problemsSolved": 3, "ranking": 1532, "rating": 1556.099, "contest": {"title": "Weekly Contest 494", "startTime": 1774146600}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 1, "ranking": 16834, "rating": 1549.485, "contest": {"title": "Weekly Contest 495", "startTime": 1774751400}}, {"attended": true, "trendDirection": "DOWN", "problemsSolved": 1, "ranking": 15151, "rating": 1544.366, "contest": {"title": "Weekly Contest 496", "startTime": 1775356200}}]']

user = extract_features(data)

print(user)

predict_user(user)





Model loaded successfully
{'Rank': 1, 'Username': 'SAIKRISHNAKANTHS', 'Score': 18, 'CurrConSolved': 3, 'time_taken': 770, 'total_fail': 3, 'Country': 'India', 'Profile_rank': 217712, 'PostViewCnt': 0, 'Reputation': 0, 'Total_solved': 462, 'Easy': 204, 'Medium': 198, 'Hard': 60, 'ContestCnt': 27, 'CurrRating': 1544.366385991523, 'rating_first': 1486.496, 'GlobalRank': 282046, 'active_days': 518}


{'lof_score': np.float64(2.5222),
 'lof_normalized': 0,
 'rule_score': 0,
 'final_score': 0.0,
 'suspicious': np.False_,
 'label': 'CLEAN',
 'triggered_rules': []}